# 🏥 Medical LLM Assistant — Colab Notebook

Notebook self-contained para rodar o sistema no Google Colab.

**Configuração rápida:**
- `USE_MOCK_LLM = 'true'` → sem GPU, respostas simuladas realistas
- `USE_MOCK_LLM = 'false'` → modelo Mistral fine-tuned via LoRA (requer GPU T4+)


In [ ]:
# Célula 1: Clone ou atualiza o repositório
# Se o repo for privado, use: https://SEU_TOKEN@github.com/...
!git clone https://github.com/felipe-huszar/medical-llm-assistant.git 2>/dev/null || \
  (cd medical-llm-assistant && git pull)

In [ ]:
# Célula 2: Instalar dependências
%cd medical-llm-assistant
!pip install -r requirements.txt -q

In [ ]:
# Célula 3: Montar Google Drive (necessário apenas se USE_MOCK_LLM=false)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print('Drive montado com sucesso.')
except ModuleNotFoundError:
    print('Não está no Colab — Drive não montado.')

In [ ]:
# Célula 4: Configuração
import os

# ⬇️  Mude para 'false' para usar o modelo real (requer GPU + LoRA no Drive)
os.environ['USE_MOCK_LLM'] = 'true'

# Caminho do adapter LoRA no Drive (ignorado se mock=true)
os.environ['MODEL_PATH'] = '/content/drive/MyDrive/medical_llm_lora'

# ID do modelo base no HuggingFace (ignorado se mock=true)
os.environ['BASE_MODEL_ID'] = 'mistralai/Mistral-7B-Instruct-v0.1'

# Diretório ChromaDB
os.environ['CHROMA_DB_PATH'] = '/content/chroma_db'

print(f"USE_MOCK_LLM = {os.environ['USE_MOCK_LLM']}")
print(f"MODEL_PATH   = {os.environ['MODEL_PATH']}")

In [ ]:
# Célula 5: Adicionar raiz do repo ao sys.path
import sys
sys.path.insert(0, '/content/medical-llm-assistant')
print('sys.path configurado.')

In [ ]:
# Célula 6: Teste rápido do pipeline (sem UI)
from src.graph.pipeline import run_consultation
from src.rag.patient_rag import seed_from_file

# Popular pacientes de exemplo
n = seed_from_file('data/patients_seed.json')
print(f'Pacientes seed inseridos: {n}')

# Rodar consulta de teste
result = run_consultation(
    cpf='123.456.789-00',
    doctor_question='Paciente relata dores abdominais ao evacuar há 3 semanas, com sangue nas fezes ocasionalmente. Quais diagnósticos e exames considerar?',
)

print('\n' + '='*60)
print(result['final_answer'])

In [ ]:
# Célula 7: Lançar interface Gradio
from app import demo
demo.launch(share=True)